In [17]:
# Day13: NLP/ML 基础 - Tokenizer
from transformers import AutoTokenizer
import pandas as pd

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# 选择 tokenizer（中文推荐 bert-base-chinese 或 GLM 对应的）  下载一个中文分词器（把汉字拆成计算机能吃的数字 ID）
#大模型（GLM、Qwen、Llama）第一步永远是 Tokenization。没有这一步，文字永远进不了神经网络。
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

# 示例文本（从 rules.md 复制一段）
text = """
## STAR 方法核心原则
Situation: 简要描述背景和情境
Task: 明确你的任务和目标
Action: 详细说明你采取的具体行动
Result: 用数据量化成果
"""

# 分词
#把一段文字变成两个张量（input_ids + attention_mask）
# pt = PyTorch 张量格式（后面要喂给 nn.Module）
# padding：把短句子补齐到相同长度（批处理必须）
# truncation：超长就截断（简历段落很少超128）
tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)

print("Tokens:", tokens['input_ids'])
print("Token count:", len(tokens['input_ids'][0]))

# 干什么：把数字变回文字（方便你检查分词是否合理）
#  ## STAR 方法 ... [SEP] 就是 BERT 类模型的标准格式
print("Decoded:", tokenizer.decode(tokens['input_ids'][0]))

# 批量处理多段文本
texts = ["用量化指标写工作经历", "STAR 方法写行动部分", "教育背景模板"]
batch = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
print("Batch shape:", batch['input_ids'].shape)



# ───────────────────────────────────────────────────────────────
#  准备 Dataset 类：把 DataFrame 变成 PyTorch 可以批量读取的格式
# ───────────────────────────────────────────────────────────────

import torch
from torch.utils.data import Dataset

class SimpleTextDataset(Dataset):
    """
    自定义 Dataset，把文本 → token → label 这一套流程封装起来
    这样 DataLoader 才能方便地批量喂数据给模型
    """
    def __init__(self, df, tokenizer, max_length=96):
        """
        df: 包含 'text' 和 'label' 两列的 pandas DataFrame
        tokenizer: 已经加载好的 tokenizer
        max_length: 统一截断/填充到的长度（简历段落通常很短，96 够用）
        """
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        """ 返回数据集总共有多少条数据 """
        return len(self.texts)

    def __getitem__(self, idx):
        """
        根据索引 idx 返回第 idx 条数据
        返回值是一个 dict，包含 input_ids 和 label
        后续模型训练时会直接从这里取数据
        """
        text = self.texts[idx]
        label = self.labels[idx]

        # 用 tokenizer 处理单条文本
        encoding = self.tokenizer(
            text,
            padding='max_length',       # 短的补 0
            truncation=True,            # 长的截断
            max_length=self.max_length,
            return_tensors='pt'         # 返回 PyTorch 张量
        )

        # squeeze(0) 把 [1, seq_len] → [seq_len]
        # 因为我们训练时 batch 会在 DataLoader 里自动加 batch 维度
        input_ids = encoding['input_ids'].squeeze(0)

        return {
            'input_ids': input_ids,
            'label': torch.tensor(label, dtype=torch.long)
        }


# ───────────────────────────────────────────────────────────────
#                  现在来创建数据集实例
# ───────────────────────────────────────────────────────────────

# 先确保你已经有 df 了（前面步骤应该已经准备好）
# 如果还没有 df，可以用下面这几行快速创建一个最小测试版

data = [
    # 规则段落 (0) - 多加几条
    {"text": "使用 STAR 方法描述工作经历：Situation、Task、Action、Result。", "label": 0},
    {"text": "量化成果必须使用具体数字，例如增长 30%、完成率 95%。", "label": 0},
    {"text": "避免使用主观形容词，如优秀、卓越，应改为可量化的表达。", "label": 0},
    {"text": "行动部分要写出你具体做了什么，而不是团队做了什么。", "label": 0},
    {"text": "结果要用数字证明影响力，比如提升了多少百分比。", "label": 0},
    {"text": "简历每条经历建议控制在 1–2 行。", "label": 0},
    {"text": "每句话都回答“产生了什么业务价值？", "label": 0},

    # 模板段落 (1)
    {"text": "教育背景：学校名称 | 专业 | GPA/排名 | 在校时间", "label": 1},
    {"text": "工作经历模板：公司名称 | 职位 | 时间 | 主要职责（建议 3-5 条）", "label": 1},
    {"text": "项目经历模板：项目名称 - 角色 - 时间 - 关键成果（量化）", "label": 1},
    {"text": "技能部分模板：按熟练度分组，如熟练 / 熟悉 / 了解", "label": 1},
    {"text": "个人简介模板：一句话概述职业方向 + 核心优势", "label": 1},
    

    # 用户经历段落 (2) - 模拟真实简历原始描述
    {"text": "2022.09 - 至今 腾讯科技有限公司 产品经理", "label": 2},
    {"text": "负责宁夏、西藏两大省份的电子税务局移动端系统，面向千万级纳税人提供掌上办税服务，涵盖申报缴款、发票管理、社保缴纳、发票代开等核心办税场景。", "label": 2},
    {"text": "2021.7 - 2022.8 字节跳动 数据分析师", "label": 2},
    {"text": "参与推荐算法优化，CTR 提升 15%", "label": 2},
    {"text": "大学期间担任学生会主席，组织活动 20+ 场", "label": 2},
    {"text": "实习期间协助市场推广，新增用户 8 万", "label": 2},
    {"text": "面向B2B交易场景的智能开票小程序，解决企业批量开票、发票自动匹配、实时状态追踪等痛点。", "label": 2},
    
]

import pandas as pd
df = pd.DataFrame(data)

# 创建 Dataset
dataset = SimpleTextDataset(df, tokenizer, max_length=96)

print("数据集总条数：", len(dataset))
print("取第一条数据看看：")
print(dataset[0])

/Users/eval/miniconda3/envs/ai-agent/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokens: tensor([[  101,   108,   108,  9012,  3175,  3791,  3417,  2552,  1333,  1156,
         10883, 10455,  8794,   131,  5042,  6206,  2989,  6835,  5520,  3250,
          1469,  2658,  1862,  8346,  8998,   131,  3209,  4802,   872,  4638,
           818,  1218,  1469,  4680,  3403, 10805,   131,  6422,  5301,  6432,
          3209,   872,  7023,  1357,  4638,  1072,   860,  6121,  1220, 13170,
           131,  4500,  3144,  2945,  7030,  1265,  2768,  3362,   102]])
Token count: 59
Decoded: [CLS] # # star 方 法 核 心 原 则 situation : 简 要 描 述 背 景 和 情 境 task : 明 确 你 的 任 务 和 目 标 action : 详 细 说 明 你 采 取 的 具 体 行 动 result : 用 数 据 量 化 成 果 [SEP]
Batch shape: torch.Size([3, 12])
数据集总条数： 19
取第一条数据看看：
{'input_ids': tensor([  101,   886,  4500,  9012,  3175,  3791,  2989,  6835,  2339,   868,
         5307,  1325,  8038, 10883, 10455,  8794,   510,  8346,  8998,   510,
        10805,   510, 13170,   511,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,   

In [19]:
from torch.utils.data import DataLoader

# batch_size 可以先设小一点，样本少的时候 2~4 比较合适
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print("DataLoader 创建成功，迭代一次看看：")
for batch in dataloader:
    print("batch['input_ids'].shape:", batch['input_ids'].shape)
    print("batch['label']:", batch['label'])
    break  # 只看第一个 batch 就行

DataLoader 创建成功，迭代一次看看：
batch['input_ids'].shape: torch.Size([4, 96])
batch['label']: tensor([0, 2, 1, 2])


In [20]:
import torch.nn as nn

class SimpleTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=3):
        super().__init__()
        # 词嵌入层：把 token id 变成向量
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # 两个全连接层
        self.fc1 = nn.Linear(embed_dim, 64)
        self.fc2 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids):
        # input_ids: [batch_size, seq_len]
        embeds = self.embedding(input_ids)          # → [batch, seq_len, embed_dim]
        pooled = embeds.mean(dim=1)                 # 平均池化 → [batch, embed_dim]
        x = torch.relu(self.fc1(pooled))
        x = self.dropout(x)
        logits = self.fc2(x)                        # → [batch, 3]
        return logits

# 创建模型实例
model = SimpleTextClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    num_classes=3  # 规则 / 模板 / 用户经历
)

print(model)

SimpleTextClassifier(
  (embedding): Embedding(21128, 128)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [21]:
# ───────────────────────────────────────────────────────────────
#                    训练循环（让模型学习分类）
# ───────────────────────────────────────────────────────────────

import torch.optim as optim
from torch.nn import CrossEntropyLoss

# 损失函数 + 优化器
criterion = CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)   # lr 可以稍大一点，因为数据很少

# 训练参数
epochs = 20   # 因为样本只有6条，epoch 多一点才能看到收敛
print_every = 3

model.train()   # 进入训练模式（启用 dropout）

print("开始训练...")
for epoch in range(epochs):
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch in dataloader:
        input_ids = batch['input_ids']
        labels = batch['label']
        
        optimizer.zero_grad()
        logits = model(input_ids)               # 前向传播
        loss = criterion(logits, labels)        # 计算交叉熵损失
        loss.backward()                         # 反向传播
        optimizer.step()                        # 更新参数
        
        total_loss += loss.item()
        
        # 顺便统计准确率（方便观察）
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total if total > 0 else 0
    
    if (epoch + 1) % print_every == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:3d}/{epochs} | "
              f"Loss: {avg_loss:.4f} | "
              f"Accuracy: {accuracy:.3f} ({correct}/{total})")

print("训练结束")

开始训练...
Epoch   3/20 | Loss: 1.1148 | Accuracy: 0.421 (8/19)
Epoch   6/20 | Loss: 1.1263 | Accuracy: 0.421 (8/19)
Epoch   9/20 | Loss: 1.1103 | Accuracy: 0.316 (6/19)
Epoch  12/20 | Loss: 1.0524 | Accuracy: 0.474 (9/19)
Epoch  15/20 | Loss: 1.0815 | Accuracy: 0.368 (7/19)
Epoch  18/20 | Loss: 1.0420 | Accuracy: 0.368 (7/19)
Epoch  20/20 | Loss: 1.0643 | Accuracy: 0.526 (10/19)
训练结束


In [22]:
print("当前类别分布：")
print(df['label'].value_counts())
print("\n每个类别的占比：")
print(df['label'].value_counts(normalize=True).round(3))

当前类别分布：
label
0    7
2    7
1    5
Name: count, dtype: int64

每个类别的占比：
label
0    0.368
2    0.368
1    0.263
Name: proportion, dtype: float64


In [9]:
# ───────────────────────────────────────────────────────────────
#               推理函数：给任意文本预测类别
# ───────────────────────────────────────────────────────────────

def classify_section(text, model, tokenizer, max_length=96):
    """
    输入一段文本，输出预测的类别
    返回：(预测类别名称, 原始 logits)
    """
    model.eval()   # 进入评估模式（关闭 dropout）
    with torch.no_grad():
        encoding = tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids']   # [1, seq_len]
        
        logits = model(input_ids)           # [1, 3]
        probs = torch.softmax(logits, dim=1)[0]   # 概率分布
        pred_idx = torch.argmax(logits, dim=1).item()
    
    label_map = {
        0: "规则段落",
        1: "模板段落",
        2: "用户经历段落"
    }
    
    return label_map[pred_idx], probs.tolist()


# ───────────────────────────────────────────────────────────────
#                        测试几条新文本
# ───────────────────────────────────────────────────────────────

test_cases = [
    "量化成果时尽量使用具体数字，例如增长30%、完成率95%。",
    "工作经历模板：公司名称 | 职位 | 时间 | 主要职责（3-5条）",
    "2023.06 - 至今 字节跳动 高级算法工程师",
    "避免使用模糊词如'显著提升'，应改为可量化的表达",
    "项目经历：负责推荐系统优化，CTR 提升 18%"
]

print("测试结果：")
for text in test_cases:
    category, probs = classify_section(text, model, tokenizer)
    print(f"文本：{text}")
    print(f"预测：{category}")
    print(f"概率分布：[规则 {probs[0]:.3f}, 模板 {probs[1]:.3f}, 经历 {probs[2]:.3f}]")
    print("-" * 60)

测试结果：
文本：量化成果时尽量使用具体数字，例如增长30%、完成率95%。
预测：规则段落
概率分布：[规则 0.364, 模板 0.304, 经历 0.333]
------------------------------------------------------------
文本：工作经历模板：公司名称 | 职位 | 时间 | 主要职责（3-5条）
预测：规则段落
概率分布：[规则 0.354, 模板 0.309, 经历 0.336]
------------------------------------------------------------
文本：2023.06 - 至今 字节跳动 高级算法工程师
预测：规则段落
概率分布：[规则 0.361, 模板 0.301, 经历 0.338]
------------------------------------------------------------
文本：避免使用模糊词如'显著提升'，应改为可量化的表达
预测：规则段落
概率分布：[规则 0.373, 模板 0.298, 经历 0.329]
------------------------------------------------------------
文本：项目经历：负责推荐系统优化，CTR 提升 18%
预测：规则段落
概率分布：[规则 0.362, 模板 0.301, 经历 0.337]
------------------------------------------------------------
